# API Data Inspection

**Purpose.** Before designing the lakehouse (bronze, silver, gold), inspect the candidate data sources from a Databricks Free Edition notebook and measure how the APIs actually behave: limits, edge cases and data quality. Every design decision on ingestion and data quality rules is grounded in the results below.

**Environment.** Databricks Free Edition, serverless compute, Python. The original exploratory run took place on 24 October 2026; the full analysis is in `source_analysis.md`.

**Chosen source.** UK Carbon Intensity API (NESO), regional data at 30 minute granularity.

> **Attribution.** Data: Carbon Intensity API, National Energy System Operator (NESO), licensed under CC BY 4.0 and used under the [API Terms of Use](https://github.com/carbon-intensity/terms). This project is not affiliated with or endorsed by NESO.

**Notebook outline**

1. Reachability of the candidate sources
2. UK Carbon Intensity: first look at the regional endpoint
3. Maximum date range per call
4. Depth of the regional history
5. All regions in a single call
6. Data quality checks on one day

## 1. Reachability of the candidate sources

Free Edition only allows outbound traffic to a set of trusted domains, and the list is not published. The only reliable test is calling each API from a notebook.

### 1.1 Open-Meteo (weather)

Candidate for a later phase (weather versus generation mix). Terms of use still under review: only a minimal sample of the output is printed.

In [0]:
import requests

r = requests.get(
    "https://api.open-meteo.com/v1/forecast?latitude=41.9&longitude=12.5&hourly=temperature_2m",
    timeout=20,
)
print(r.status_code, r.text[:200])

200 {"latitude":41.895107,"longitude":12.526347,"generationtime_ms":0.5824565887451172,"utc_offset_seconds":0,"timezone":"GMT","timezone_abbreviation":"GMT","elevation":58.0,"hourly_units":{"time":"iso860


### 1.2 Terna Download Center (Italian electricity load): discarded

The endpoint answered with HTTP 200, but **no licence allowing reuse of the data was found**: the Terms of Use available cover only the developer portal. Since this project only uses open data with a clear licence, the source was discarded.

The cell is kept below as non executable code, to document the decision without republishing the data. One technical quirk was also observed: the CSV uses the comma both as field separator and as decimal separator, so a value written like `1234,500000` (an illustrative format, not real data) cannot be parsed reliably.

```python
import requests
url = ("https://dati.terna.it/api/sitecore/dati/downloadcenter/records"
       "?f=csv&filterDataset=TotalLoad&filterYear=2025&db=dati&pageSize=5")
r = requests.get(url, timeout=30)
print(r.status_code)
print(r.text[:600])
```

### 1.3 Energy-Charts, Fraunhofer ISE (European electricity data)

Kept as a possible future extension to Italy and other European countries. Data are national, not regional. Terms, as published by the provider: non commercial and journalistic use, attribution "Energy-Charts / Fraunhofer ISE"; the official legal text is still to be confirmed.

In [0]:
r = requests.get(
    "https://api.energy-charts.info/public_power",
    params={"country": "it", "start": "2026-10-01", "end": "2026-10-02"},
    timeout=30,
)
print(r.status_code)
j = r.json()
print("top level keys:", list(j.keys()))
print("series (name, number of values):")
for p in j["production_types"]:
    print(" ", p["name"], len(p["data"]))

200
top level keys: ['unix_seconds', 'production_types', 'deprecated']
series (name, number of values):
  Hydro pumped storage consumption 192
  Battery Consumption 192
  Cross border electricity trading 192
  Hydro Run-of-River 192
  Biomass 192
  Fossil hard coal 192
  Fossil oil 192
  Fossil gas 192
  Geothermal 192
  Hydro water reservoir 192
  Hydro pumped storage 192
  Battery 192
  Others 192
  Wind offshore 192
  Wind onshore 192
  Solar 192
  Load 192
  Residual load 192
  Renewable share of load 192
  Renewable share of generation 192


Same request for several countries: does the `Load` series have the same granularity everywhere? Expected 192 values for two days at 15 minutes.

In [0]:
for c in ["it", "de", "fr", "es", "at", "ch", "pl"]:
    r = requests.get(
        "https://api.energy-charts.info/public_power",
        params={"country": c, "start": "2026-10-01", "end": "2026-10-02"},
        timeout=30,
    )
    result = None
    if r.status_code == 200:
        series = {p["name"]: p["data"] for p in r.json()["production_types"]}
        load = series.get("Load", [])
        result = (len(load), sum(v is not None for v in load))
    print(c, r.status_code, "(values, non null):", result)

it 200 (values, non null): (192, 192)
de 200 (values, non null): (192, 192)
fr 200 (values, non null): (192, 192)
es 200 (values, non null): (192, 192)
at 200 (values, non null): (192, 192)
ch 200 (values, non null): (48, 48)
pl 200 (values, non null): (192, 192)


**Finding.** In the original run Switzerland (`ch`) returned 48 hourly values while the other countries returned 192 values at 15 minutes: granularity is not uniform across countries.

## 2. UK Carbon Intensity: first look at the regional endpoint

`/regional` returns the current half hour for every region: carbon intensity forecast (gCO2/kWh), an intensity index and the generation mix by fuel.

In [0]:
r = requests.get("https://api.carbonintensity.org.uk/regional", timeout=30)
print("HTTP", r.status_code)
period = r.json()["data"][0]
print("half hour:", period["from"], "to", period["to"])
for x in period["regions"]:
    first_three = [(m["fuel"], m["perc"]) for m in x["generationmix"]][:3]
    print(x["regionid"], x["shortname"], x["intensity"], first_three)

HTTP 200
half hour: 2026-10-24T17:30Z to 2026-10-24T18:00Z
1 North Scotland {'forecast': 0, 'index': 'very low'} [('biomass', 0), ('coal', 0), ('imports', 0)]
2 South Scotland {'forecast': 0, 'index': 'very low'} [('biomass', 0), ('coal', 0), ('imports', 0)]
3 North West England {'forecast': 26, 'index': 'low'} [('biomass', 0), ('coal', 0), ('imports', 1.2)]
4 North East England {'forecast': 0, 'index': 'very low'} [('biomass', 0), ('coal', 0), ('imports', 7.6)]
5 Yorkshire {'forecast': 47, 'index': 'low'} [('biomass', 12.6), ('coal', 0), ('imports', 2.3)]
6 North Wales & Merseyside {'forecast': 70, 'index': 'low'} [('biomass', 0), ('coal', 0), ('imports', 1.3)]
7 South Wales {'forecast': 273, 'index': 'very high'} [('biomass', 0), ('coal', 0), ('imports', 0)]
8 West Midlands {'forecast': 77, 'index': 'low'} [('biomass', 0), ('coal', 0), ('imports', 1)]
9 East Midlands {'forecast': 155, 'index': 'moderate'} [('biomass', 6.7), ('coal', 0), ('imports', 1.2)]
10 East England {'forecast': 

**Findings**

* 18 entries are returned, but ids 15 to 18 (England, Scotland, Wales, GB) are **aggregates** of the regions 1 to 14. Comparing or summing all 18 rows would count the same territory twice, much like mixing detail rows and subtotal rows of a SQL `GROUP BY ROLLUP`. Only ids 1 to 14 are used for comparisons.
* Regions are electricity distribution areas, not administrative boundaries.
* The documentation flags regional data as beta.

## 3. Maximum date range per call

The documentation states a 14 day limit for the **national** endpoint `/intensity/{from}/{to}`, but no limit for the regional endpoints. The limit is therefore measured.

Helper function used in sections 3 and 4. It queries one region over a date range and pauses one second after each call, because the Terms of Use state that a rate limit applies (its value is not published).

In [0]:
import time
from datetime import datetime, timedelta, timezone

BASE = "https://api.carbonintensity.org.uk"

def regional(start, end, region_id):
    url = (f"{BASE}/regional/intensity/"
           f"{start:%Y-%m-%dT%H:%MZ}/{end:%Y-%m-%dT%H:%MZ}/regionid/{region_id}")
    r = requests.get(url, headers={"Accept": "application/json"}, timeout=30)
    time.sleep(1)  # be gentle: the Terms of Use mention a rate limit
    if r.status_code != 200:
        return r.status_code, None, r.text[:200]
    body = r.json()["data"]
    # Verified structure: a single region object with its list of half hours
    body = body[0] if isinstance(body, list) else body
    return r.status_code, body.get("data", []), body.get("shortname")

### 3.1 Increasing ranges, region 13 (London)

Expected rows: days × 48 half hours.

In [0]:
end = datetime(2026, 10, 1, tzinfo=timezone.utc)
for days in [1, 7, 14, 15, 31]:
    status, rows, info = regional(end - timedelta(days=days), end, 13)
    n = len(rows) if rows is not None else None
    print(f"{days:>3} days -> HTTP {status}, rows {n}, expected {days * 48}, region: {info}")

  1 days -> HTTP 200, rows 49, expected 48, region: London
  7 days -> HTTP 200, rows 337, expected 336, region: London
 14 days -> HTTP 200, rows 673, expected 672, region: London
 15 days -> HTTP 200, rows 721, expected 720, region: London
 31 days -> HTTP 200, rows 1489, expected 1488, region: London


**Findings**

* Ranges up to 31 days are accepted: the 14 day limit does not apply to the regional endpoint.
* Every call returns **one row more** than expected. Section 4 explains why.

### 3.2 Longer ranges

Error messages are printed with `repr()`: the response body contains carriage return characters (`\r`) that would otherwise overwrite the printed line and hide the HTTP status.

In [0]:
for days in [60, 92, 183, 366]:
    status, rows, info = regional(end - timedelta(days=days), end, 13)
    n = len(rows) if rows is not None else None
    print(f"{days:>3} days -> HTTP {status}, rows {n}, detail: {repr(info)[:110]}")

 60 days -> HTTP 400, rows None, detail: '{\r"error": {\r  "code": "400 Bad Request",\r  "message": "The date range you have specified is greater than 
 92 days -> HTTP 400, rows None, detail: '{\r"error": {\r  "code": "400 Bad Request",\r  "message": "The date range you have specified is greater than 
183 days -> HTTP 400, rows None, detail: '{\r"error": {\r  "code": "400 Bad Request",\r  "message": "The date range you have specified is greater than 
366 days -> HTTP 400, rows None, detail: '{\r"error": {\r  "code": "400 Bad Request",\r  "message": "The date range you have specified is greater than 


### 3.3 The exact boundary

In [0]:
for label, delta in [("59 days", timedelta(days=59)),
                     ("60 days minus 30 minutes", timedelta(days=60, minutes=-30)),
                     ("60 days", timedelta(days=60))]:
    status, rows, info = regional(end - delta, end, 13)
    n = len(rows) if rows is not None else None
    print(f"{label:<26} -> HTTP {status}, rows {n}, detail: {repr(info)[:110]}")

59 days                    -> HTTP 200, rows 2833, detail: 'London'
60 days minus 30 minutes   -> HTTP 200, rows 2880, detail: 'London'
60 days                    -> HTTP 400, rows None, detail: '{\r"error": {\r  "code": "400 Bad Request",\r  "message": "The date range you have specified is greater than 


**Findings**

* 59 days: 2,833 rows (59 × 48 + 1).
* 60 days minus 30 minutes: 2,880 rows, that is exactly 60 days of data.
* 60 days: HTTP 400, "The date range you have specified is greater than 60 days".

The real limit is **60 days of effective range**, measured on the window **after** the API has extended it by 30 minutes at the start (see section 4). Monthly ingestion blocks stay well below the limit.

## 4. Depth of the regional history

One day is requested at sample dates, region 13. The first row of each response is printed.

In [0]:
probe_dates = ["2017-09-01", "2018-05-01", "2018-06-01", "2019-01-01",
               "2020-01-01", "2022-01-01", "2024-01-01", "2026-01-01"]
for d in probe_dates:
    start = datetime.fromisoformat(d).replace(tzinfo=timezone.utc)
    status, rows, info = regional(start, start + timedelta(days=1), 13)
    n = len(rows) if rows is not None else None
    first = rows[0]["from"] if rows else None
    print(f"{d} -> HTTP {status}, rows {n} (expected 48), first row from {first}, region: {info}")

2017-09-01 -> HTTP 200, rows 0 (expected 48), first row from None, region: None
2018-05-01 -> HTTP 200, rows 0 (expected 48), first row from None, region: None
2018-06-01 -> HTTP 200, rows 49 (expected 48), first row from 2018-05-31T23:30Z, region: London
2019-01-01 -> HTTP 200, rows 49 (expected 48), first row from 2018-12-31T23:30Z, region: London
2020-01-01 -> HTTP 200, rows 49 (expected 48), first row from 2019-12-31T23:30Z, region: London
2022-01-01 -> HTTP 200, rows 49 (expected 48), first row from 2021-12-31T23:30Z, region: London
2024-01-01 -> HTTP 200, rows 49 (expected 48), first row from 2023-12-31T23:30Z, region: London
2026-01-01 -> HTTP 200, rows 49 (expected 48), first row from 2025-12-31T23:30Z, region: London


**Findings**

* **History starts between 1 May 2018 and 1 June 2018**: more than eight years of regional data.
* **"No data" is not an error**: for dates without data the API answers HTTP 200 with zero rows. The ingestion must record this case explicitly instead of treating it as a silent success.
* **The extra half hour.** Requesting a day from 00:00 returns a first row starting at 23:30 of the **previous** day. The API also returns the half hour that **ends** at the requested start. Consequence: with consecutive load blocks (September, then October), the half hour at the boundary arrives **twice**. This measured behaviour is the concrete reason why the silver layer must deduplicate on (region, period start).
* Caution: only one day every one or two years was sampled. Gaps in between will surface only with the full load, through a completeness check in silver.

## 5. All regions in a single call

The same endpoint **without** `regionid` returns every region for the requested range. It is the equivalent of removing a filter predicate from a SQL `WHERE` clause.

In [0]:
start = datetime(2026, 9, 1, tzinfo=timezone.utc)
url = (f"{BASE}/regional/intensity/"
       f"{start:%Y-%m-%dT%H:%MZ}/{start + timedelta(days=1):%Y-%m-%dT%H:%MZ}")
r = requests.get(url, headers={"Accept": "application/json"}, timeout=30)
print("HTTP", r.status_code)
body = r.json()["data"]
print("type of data:", type(body).__name__, "| length:", len(body))
print("keys of the first element:", list(body[0].keys()))
print("first element, truncated:", str(body[0])[:400])

HTTP 200
type of data: list | length: 49
keys of the first element: ['from', 'to', 'regions']
first element, truncated: {'from': '2026-08-31T23:30Z', 'to': '2026-09-01T00:00Z', 'regions': [{'regionid': 1, 'dnoregion': 'Scottish Hydro Electric Power Distribution', 'shortname': 'North Scotland', 'intensity': {'forecast': 0, 'index': 'very low'}, 'generationmix': [{'fuel': 'biomass', 'perc': 0}, {'fuel': 'coal', 'perc': 0}, {'perc': 0, 'fuel': 'imports'}, {'fuel': 'gas', 'perc': 0}, {'perc': 0, 'fuel': 'nuclear'}, {'p


**Findings**

* `data` is a list of 49 half hours (48 plus the extra one at the start).
* The structure is nested on three levels: **half hour → region → fuel**. With `regionid` the order is the opposite (region → half hour).

```
data (list of half hours)
 └─ from, to, regions (list of 18 regions)
              └─ regionid, dnoregion, shortname,
                 intensity {forecast, index},
                 generationmix (list of fuels)
                               └─ fuel, perc
```

* `dnoregion` holds the name of the distribution network operator: material for a small region dimension table.
* Inside `generationmix` the key order changes between elements (`fuel` before `perc` or the opposite): fields must always be read by name, never by position.
* **One call returns all regions**, so a monthly load needs 1 call instead of 14: 36 calls for three years of history instead of 504.

## 6. Data quality checks on one day (1 September 2026)

Five checks on the response of section 5, each with its SQL equivalent. They become the starting point for the silver data quality rules.

In [0]:
from collections import Counter

# Check 1: regions per half hour  (SELECT COUNT(*) ... GROUP BY period)
regions_per_period = Counter(len(p["regions"]) for p in body)
print("regions per half hour:", dict(regions_per_period))

# Check 2: distinct region ids  (SELECT DISTINCT regionid)
print("region ids:", sorted({reg["regionid"] for p in body for reg in p["regions"]}))

# Check 3: fuels  (SELECT fuel, COUNT(*) ... GROUP BY fuel)
fuels = Counter(m["fuel"] for p in body for reg in p["regions"] for m in reg["generationmix"])
print("fuels:", dict(fuels))

regions per half hour: {18: 49}
region ids: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18]
fuels: {'biomass': 882, 'coal': 882, 'imports': 882, 'gas': 882, 'nuclear': 882, 'other': 882, 'hydro': 882, 'solar': 882, 'wind': 882}


In [0]:
# Check 4: mix total per (half hour, region)  (SUM(perc) ... GROUP BY period, regionid)
totals = [round(sum(m["perc"] for m in reg["generationmix"]), 1)
          for p in body for reg in p["regions"]]
print("mix total: min", min(totals), "max", max(totals))
outside = [t for t in totals if abs(t - 100) > 1]
print("totals more than 1 point away from 100:", len(outside), "of", len(totals))

# Check 5: missing intensity values  (COUNT(*) WHERE forecast IS NULL)
missing = sum(1 for p in body for reg in p["regions"]
              if reg["intensity"].get("forecast") is None)
print("null forecasts:", missing)

mix total: min 99.8 max 100.1
totals more than 1 point away from 100: 0 of 882
null forecasts: 0


**Findings (original run)**

| Check | Result | Reading |
|---|---|---|
| Regions per half hour | `{18: 49}` | No region missing in any half hour |
| Region ids | 1 to 18 | No gaps, no unexpected ids |
| Fuels | 9 fuels, 882 times each | 882 = 49 × 18: every region reports all 9 fuels in every half hour, zeros included |
| Mix total | min 99.8, max 100.1; 0 of 882 outside ±1 | Deviation caused by rounding of the single percentages |
| Null forecasts | 0 | No missing values |

Fuels: biomass, coal, imports, gas, nuclear, other, hydro, solar, wind.

The mix total is computed **within** one region, so including the aggregates 15 to 18 does not change any single total. Double counting appears only when summing **across** regions.

**Caution:** these checks cover a single day. Thresholds must be confirmed on the full history, which is why the silver layer will separate **blocking rules** (the row is rejected) from **warning rules** (the row is kept and flagged).

## Summary: documented versus measured behaviour

| Aspect | Documentation | Measured |
|---|---|---|
| Maximum range per call (regional) | 14 days, stated for the national endpoint | 60 days of effective range |
| Rows returned | not stated | one extra half hour at the start |
| Missing data | not stated | HTTP 200 with zero rows |
| Start of regional history | not stated | between 1 May and 1 June 2018 |
| Error format | not stated | JSON with `\r` characters, HTTP 400 for ranges too long |
| Rate limit | not stated in the docs | stated in the Terms of Use, value not published |

Next step: the analysis and design document of the lakehouse, built on these findings.